# Notebook 03 – Geomarketing-Scoring-Modell & Top-5-Standorte
**Charge & Ride Züri** | ZHAW Geomarketing FS2026

Dieses Notebook implementiert das Scoring-Modell zur Identifikation
der 5 optimalen Standorte für neue E-Bike-Ladestationen.

**Voraussetzungen:** Notebooks 01 und 02 müssen vollständig ausgeführt worden sein.

**Methodik:**
1. Supply-Analyse: 3 km-Puffer um bestehende Ladestationen
2. Demand-Analyse: KDE-Heatmap der Velofrequenzen
3. Scoring: POIs ausserhalb Versorgungszone bewerten
4. Top-5 exportieren für QGIS

**Outputs:**
- `supply_zone_3km.geojson` – Versorgungszone bestehender Ladestationen
- `alle_kandidaten_scored.geojson` – Alle bewerteten Kandidaten-POIs
- `top5_standorte.geojson` – Die 5 Topstandorte (für QGIS-Karte 3)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import contextily as cx
import folium
from folium.plugins import HeatMap

ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'

# Scoring-Gewichtungen (anpassbar – begruendung im Bericht!)
W_FREQUENZ = 0.5    # Gewicht: Velofrequenz
W_DISTANZ = 0.5     # Gewicht: Distanz zur naechsten Ladesaeule

BUFFER_RADIUS_M = 3000  # 3 km Versorgungszone

print(f'Scoring-Gewichtungen: Frequenz={W_FREQUENZ}, Distanz={W_DISTANZ}')

## 1. Daten laden (aus Notebook 01 und 02)

In [ ]:
# Alle verarbeiteten Daten laden
gdf_ladestationen = gpd.read_file(DATA_PROCESSED / 'ladestationen_zh.geojson').to_crs(epsg=2056)
gdf_gastro = gpd.read_file(DATA_PROCESSED / 'gastronomie_zh.geojson').to_crs(epsg=2056)
gdf_bahnhoefe = gpd.read_file(DATA_PROCESSED / 'bahnhoefe_zh.geojson').to_crs(epsg=2056)
gdf_kanton = gpd.read_file(DATA_PROCESSED / 'kanton_zh_boundary.geojson').to_crs(epsg=2056)
gdf_zaehlstellen = gpd.read_file(DATA_PROCESSED / 'velozaehlstellen_mit_frequenz.geojson').to_crs(epsg=2056)

print('Daten geladen:')
print(f'  Ladestationen: {len(gdf_ladestationen)}')
print(f'  Gastronomie:   {len(gdf_gastro)}')
print(f'  Bahnhoefe:     {len(gdf_bahnhoefe)}')
print(f'  Zaehlstellen:  {len(gdf_zaehlstellen)}')

## 2. Supply-Analyse: Versorgungszone (3 km-Puffer)

In [ ]:
# 3 km-Puffer um bestehende Ladestationen (in LV95: Meter!)
puffer_einzeln = gdf_ladestationen.geometry.buffer(BUFFER_RADIUS_M)

# Alle Puffer zu einem einzigen Polygon vereinen
from shapely.ops import unary_union
supply_zone = unary_union(puffer_einzeln)

gdf_supply = gpd.GeoDataFrame({'geometry': [supply_zone]}, crs='EPSG:2056')

# Speichern
gdf_supply.to_crs(epsg=4326).to_file(DATA_PROCESSED / 'supply_zone_3km.geojson', driver='GeoJSON')

# Versorgungsquote berechnen
kanton_area = gdf_kanton.geometry.area.sum()
supply_area = supply_zone.area
print(f'Versorgungsquote Kanton ZH: {supply_area/kanton_area*100:.1f}%')
print('Supply-Zone gespeichert.')

## 3. Kandidaten-POIs zusammenstellen und filtern

In [ ]:
# Kandidaten = Gastronomie + Bahnhöfe
gdf_gastro['poi_typ'] = 'gastronomie'
gdf_bahnhoefe['poi_typ'] = 'bahnhof'

# Gemeinsamen GeoDataFrame erstellen
cols_needed = ['geometry', 'name', 'poi_typ']
gdf_kandidaten = pd.concat([
    gdf_gastro[[c for c in cols_needed if c in gdf_gastro.columns]],
    gdf_bahnhoefe[[c for c in cols_needed if c in gdf_bahnhoefe.columns]]
], ignore_index=True)
gdf_kandidaten = gpd.GeoDataFrame(gdf_kandidaten, crs='EPSG:2056')

# Nur POIs AUSSERHALB der Versorgungszone behalten
gdf_kandidaten['in_supply_zone'] = gdf_kandidaten.geometry.within(supply_zone)
gdf_weisse_flecken = gdf_kandidaten[~gdf_kandidaten['in_supply_zone']].copy()

print(f'Alle Kandidaten:            {len(gdf_kandidaten)}')
print(f'Ausserhalb Versorgungszone: {len(gdf_weisse_flecken)}')

## 4. Scoring-Kriterium 1: Distanz zur nächsten Ladesäule

In [ ]:
from shapely.ops import nearest_points

# Distanz zur naechsten bestehenden Ladesaeule berechnen
ladesaeule_union = unary_union(gdf_ladestationen.geometry)

gdf_weisse_flecken['distanz_ladesaeule_m'] = gdf_weisse_flecken.geometry.apply(
    lambda p: p.distance(ladesaeule_union)
)

print(f'Max. Distanz: {gdf_weisse_flecken["distanz_ladesaeule_m"].max():.0f} m')
print(f'Mittlere Distanz: {gdf_weisse_flecken["distanz_ladesaeule_m"].mean():.0f} m')

## 5. Scoring-Kriterium 2: Nähe zu Velo-Hotspots (KDE)

In [ ]:
# Kernel Density Estimation (KDE) der Velozaehlstellen-Frequenzen
# Gewichtet nach mittlerer Tagesfrequenz

# Koordinaten der Zaehlstellen
coords = np.array([
    [p.x for p in gdf_zaehlstellen.geometry],
    [p.y for p in gdf_zaehlstellen.geometry]
])

# KDE erstellen (gewichtet nach Frequenz)
# TODO: 'mittlere_tagesfrequenz' nach tatsaechlichem Spaltennamen anpassen
weights = gdf_zaehlstellen['mittlere_tagesfrequenz'].fillna(0).values
kde = gaussian_kde(coords, weights=weights)

# KDE-Wert fuer jeden Kandidaten berechnen
kandidaten_coords = np.array([
    [p.x for p in gdf_weisse_flecken.geometry],
    [p.y for p in gdf_weisse_flecken.geometry]
])

gdf_weisse_flecken['kde_velo_score'] = kde(kandidaten_coords)
print('KDE-Score berechnet.')

## 6. Gesamt-Score berechnen und Top-5 ausgeben

In [ ]:
# Min-Max-Normalisierung (0-1)
def normalize(series):
    """Min-Max-Normalisierung auf [0, 1]"""
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return series * 0  # Alle gleich -> alle 0
    return (series - min_val) / (max_val - min_val)

gdf_weisse_flecken['norm_distanz'] = normalize(gdf_weisse_flecken['distanz_ladesaeule_m'])
gdf_weisse_flecken['norm_kde']     = normalize(gdf_weisse_flecken['kde_velo_score'])

# Gewichteter Gesamt-Score
gdf_weisse_flecken['gesamt_score'] = (
    W_FREQUENZ * gdf_weisse_flecken['norm_kde'] +
    W_DISTANZ  * gdf_weisse_flecken['norm_distanz']
)

# Sortieren und Top-5
gdf_scored = gdf_weisse_flecken.sort_values('gesamt_score', ascending=False).copy()
gdf_top5 = gdf_scored.head(5).copy()

print('TOP 5 STANDORTE:')
print(gdf_top5[['name', 'poi_typ', 'gesamt_score', 'distanz_ladesaeule_m', 'kde_velo_score']]
      .to_string(index=False))

## 7. Ergebnisse speichern

In [ ]:
# Alle Kandidaten speichern
gdf_scored.to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'alle_kandidaten_scored.geojson', driver='GeoJSON'
)

# Top 5 speichern
gdf_top5.to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'top5_standorte.geojson', driver='GeoJSON'
)

print('Ergebnisse gespeichert.')
print('-> Naechster Schritt: GeoJSONs in QGIS laden und Karten layouten!')

## 8. Interaktive Karte (Folium)

In [ ]:
# Interaktive Folium-Karte mit allen Ergebnissen
center = [47.4, 8.65]  # Zentrum Kanton Zuerich
m = folium.Map(location=center, zoom_start=10, tiles='OpenStreetMap')

# Heatmap der Velofrequenzen
heat_data = [
    [row.geometry.y, row.geometry.x, row['mittlere_tagesfrequenz']]
    for _, row in gdf_zaehlstellen.to_crs(epsg=4326).iterrows()
    if not pd.isna(row.get('mittlere_tagesfrequenz', np.nan))
]
HeatMap(heat_data, name='Velo-Hotspots', min_opacity=0.3).add_to(m)

# Bestehende Ladestationen
for _, row in gdf_ladestationen.to_crs(epsg=4326).iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='green', fill=True,
        popup=f"Bestehend: {row.get('name', 'k.A.')}"
    ).add_to(m)

# Top-5-Standorte
for i, (_, row) in enumerate(gdf_top5.to_crs(epsg=4326).iterrows(), 1):
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"TOP {i}: {row.get('name', 'k.A.')} | Score: {row['gesamt_score']:.3f}",
        icon=folium.Icon(color='red', icon='bolt', prefix='fa')
    ).add_to(m)

folium.LayerControl().add_to(m)

# Speichern
m.save(str(DATA_PROCESSED / 'interaktive_karte.html'))
print('Interaktive Karte gespeichert: data/processed/interaktive_karte.html')
m